# Condition classification datasets

Prepare P/N classification examples, a cluster-disjoint holdout, and training subsets by publication year. The implementation and split parameters are in `mofinder.datasets.prepare` and `configs/dataset_preparation.json`.

## Inputs and setup

Install `python -m pip install -e ".[datasets,notebook]"` from the repository root. Run the cells in order. Preparation runs locally and requires no API key. All paths below are repository-relative.

| Input | Default location | Included alternative |
| --- | --- | --- |
| Positive stage 6, CSV | `results/curation/positive/mof_extraction_1_2_3_4_5_6.csv` | `data/processed/revised/positive_stage6.csv` |
| Negative stage 6, CSV | `results/curation/negative/mof_extraction_failures_enum_1_2_3_4_5_6.csv` | `data/processed/revised/negative_stage6_v3.csv` |
| Publication years, CSV | `data/metadata/publication_years.csv` | Included; `DOI` and `Publication Year` columns. |
| Classification prompt, TXT | `prompts/training/reaction_prediction.txt` | Included. |
| Holdout conditions, JSON | `configs/dataset_forced_questions.json` | Included. |

For a run using the included full cleaned tables, change `load_settings` below to `configs/dataset_preparation_archived.json`. For a smaller test, use [the JSON preparation demo](../Demo/02_json_preparation/demo.ipynb). To prepare another dataset, select its cleaned CSVs and publication metadata in the configuration, retaining the existing column schema.

Enable `RUN_PREPARATION` after validation. The main configuration writes `results/datasets/conditions/`; the archived-input configuration writes `results/datasets/archived_conditions/`. Each output contains training and holdout JSONL, class labels, split assignments, and a summary. The packaged research files are `data/training/train.jsonl` and `data/training/holdout.jsonl`; their split records are in `data/splits/`.

Implementation: [dataset preparation and cluster splitting](../src/mofinder/datasets/prepare.py). See the [source-to-code guide](../docs/source_to_code.md) for the original workflow stages and their corresponding functions.


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "mofinder").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "mofinder").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from mofinder.display import display_paths
from mofinder.datasets.prepare import load_settings, prepare, validate_inputs

settings = load_settings(PROJECT_ROOT / "configs/dataset_preparation.json")


## Inputs

The default configuration uses positive stage 6 and negative stage 6 from curation. To use the archived positive stage 6 and negative stage `6_v3` tables, select `configs/dataset_preparation_archived.json`. Positive stage 7 trimming is optional and is not selected automatically.

Publication metadata needs `DOI` and `Publication Year`. Prompts and the 22 benchmark condition definitions are stored separately.


In [ ]:
validation = validate_inputs(settings)
print(json.dumps(display_paths(validation), indent=2))


## Prepare the split

Rows share a cluster when their primary metal precursor, complete linker set, and complete solvent set match. Matching benchmark conditions force the entire cluster into holdout; one surviving example per matched condition is protected during P/N balancing. Matching uses the eight input fields and does not impose the benchmark reference label. The summary reports unmatched questions.

P/N balancing enforces the same exact ratio in train and holdout. It does not necessarily produce a 1:1 ratio. Four-period and five-period year subsets contain training records only.


In [ ]:
RUN_PREPARATION = False
result = None
if RUN_PREPARATION:
    result = prepare(settings)
    print(json.dumps(display_paths(result["summary"]), ensure_ascii=False, indent=2))


## Inspect coverage and outputs

The full train/holdout files include rows without publication year; year subsets exclude them. `mof_ft_split_assignments.csv` records source row IDs, DOI, exact input keys, clusters, and partition assignments. Positive rows appear first in the concatenated input, followed by negative rows; source IDs are zero-based.

This step prepares training files. It does not upload data or start model fine-tuning.


In [ ]:
if result is not None:
    print(json.dumps(result["summary"]["forced_holdout"], indent=2))
    print(json.dumps(display_paths(result["summary"]["outputs"]), indent=2))
